## Airbnb Chatbot
Collaborators: Ben Ghouzi, Shawn Lokshin, Khadiatou Ly, Veronica Song

Goal: Build a chatbot that can process user's natural language queries to recommend experiences and services based on user-user filtering and selected housing location

In [1]:
import numpy as np
import pandas as pd

listings = pd.read_csv("data/listings.csv")
reviews = pd.read_csv("data/reviews.csv")
user_booking = pd.read_csv("data/user_bookings.csv")

user_booking.head()

,user_id,listing_id,city,trip_purpose,guests,nights_stayed,total_paid,rating,booking_month
0,U0145,L0290,Chicago,group,4,3,834,5,10
1,U0189,L0337,New York,leisure,2,3,768,4,7
2,U0172,L0016,Boston,leisure,2,7,1197,3,3
3,U0020,L0465,Miami,romantic,2,7,826,5,2
4,U0024,L0148,Boston,leisure,1,5,550,5,12


In [3]:
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 13.4 MB/s  0:00:00 eta 0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [4]:
# using clustering text analysis to determine sentiment from user reviews
# goal: assign a numeric rating to each users review from reviews.csv

# Install dependencies if needed:
# pip install pandas spacy nltk vaderSentiment

import spacy
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import nltk

# Download punkt tokenizer if not already
nltk.download('punkt')

# Load spaCy English model
# python -m spacy download en_core_web_sm
nlp = spacy.load("en_core_web_sm")

# Initialize sentiment analyzer
analyzer = SentimentIntensityAnalyzer()

# -------------------------------
# 1. Load Data
# -------------------------------

# Ensure column exists
assert "review_text" in reviews.columns

# -------------------------------
# 2. Phrase Extraction Function
# -------------------------------
def extract_phrases(text):
    """
    Extract meaningful phrases using noun chunks + sentence splits
    """
    doc = nlp(str(text))
    
    phrases = []
    
    # Sentence-level splitting
    for sent in doc.sents:
        phrases.append(sent.text.strip())
    
    # Add noun chunks for finer granularity
    for chunk in doc.noun_chunks:
        phrases.append(chunk.text.strip())
    
    return list(set(phrases))  # remove duplicates

# -------------------------------
# 3. Phrase Sentiment Scoring
# -------------------------------
def score_phrase(phrase):
    """
    Returns compound sentiment score (-1 to +1)
    """
    score = analyzer.polarity_scores(phrase)
    return score['compound']

# -------------------------------
# 4. Review-Level Aggregation
# -------------------------------
def compute_review_sentiment(text):
    phrases = extract_phrases(text)
    
    if not phrases:
        return 0, []
    
    phrase_scores = []
    
    for phrase in phrases:
        score = score_phrase(phrase)
        phrase_scores.append((phrase, score))
    
    # Weighted aggregation (longer phrases = higher weight)
    weighted_sum = 0
    total_weight = 0
    
    for phrase, score in phrase_scores:
        weight = len(phrase.split())
        weighted_sum += score * weight
        total_weight += weight
    
    overall_score = weighted_sum / total_weight if total_weight != 0 else 0
    
    return overall_score, phrase_scores

# -------------------------------
# 5. Apply to Dataset
# -------------------------------
results = reviews["review_text"].apply(compute_review_sentiment)

reviews["sentiment_score"] = results.apply(lambda x: x[0])
reviews["phrase_sentiments"] = results.apply(lambda x: x[1])

# -------------------------------
# 6. Optional: Label Categories
# -------------------------------
def label_sentiment(score):
    if score >= 0.05:
        return "positive"
    elif score <= -0.05:
        return "negative"
    else:
        return "neutral"

reviews["sentiment_label"] = reviews["sentiment_score"].apply(label_sentiment)

# -------------------------------
# 7. Save Results
# -------------------------------
reviews.to_csv("reviews_with_sentiment.csv", index=False)

# -------------------------------
# 8. Preview Example
# -------------------------------
print(reviews[["review_text", "sentiment_score", "sentiment_label"]].head())

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/veronicasong/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


                                         review_text  sentiment_score  \
0  Stayed here for a week and loved every minute....         0.336859   
1  Great location, just 9 minutes from Lexington ...         0.368300   
2  Beautiful apartment in a great part of New Yor...         0.354510   
3  Perfect for our business trip. Close to Centra...         0.329196   
4  Wonderful host, beautiful space in Midtown Man...         0.369892   

  sentiment_label  
0        positive  
1        positive  
2        positive  
3        positive  
4        positive  


LLM prompt used: Act like a professional data scientist. I am currently trying to build a sentiment text analysis model based off user review text stored in reviews.csv under column review_text. Please code in python a phrase-level sentiment analyser that separates out the user reviews into phrases and then assigns an overall user sentiment score to their review.

In [ ]:
# setting up user-user collaborative filtering
# connect the user sentiment score to the listing

LLM prompt used: Act like a professional data scientist. I am currently trying to build a user-user collaborative filtering function that 

In [ ]:
# setting up chat bot using ____ natural language intepretation

LLM prompt used: 